# Cybersecurity Expert System - Model Training (Semua Dataset)
Jalankan notebook ini di Google Colab atau Kaggle untuk melatih model Random Forest menggunakan semua dataset (Intel Ancaman, CVE, Domain, dan IP).

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import joblib
import os

# 1. Upload SEMUA Dataset Anda ke Colab terlebih dahulu:
# - 1_otx_threat_intel.csv
# - 2_cve_vulnerabilities.csv
# - 3_malicious_domains.csv
# - 4_malicious_ips.csv

print("Loading all datasets...")

try:
    df1 = pd.read_csv('1_otx_threat_intel.csv')[['Tags', 'Title']].rename(columns={'Tags': 'features', 'Title': 'label'})
    df2 = pd.read_csv('2_cve_vulnerabilities.csv')[['cwes', 'vulnerabilityName']].rename(columns={'cwes': 'features', 'vulnerabilityName': 'label'})
    df3 = pd.read_csv('3_malicious_domains.csv')[['Categories', 'Threat_Severity']].rename(columns={'Categories': 'features', 'Threat_Severity': 'label'})
    df4 = pd.read_csv('4_malicious_ips.csv')[['Threat_Category', 'Threat_Label']].rename(columns={'Threat_Category': 'features', 'Threat_Label': 'label'})
except FileNotFoundError as e:
    print(f"\nERROR: {e}")
    print("TOLONG PASTIKAN SEMUA FILE CSV SUDAH DI-UPLOAD KE COLAB!")
    raise

# Gabungkan semua dataset
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Filter invalid data
df = df.dropna(subset=['features', 'label'])
df = df[df['features'] != 'Unknown']
df = df[df['label'] != 'Unknown']

print("Extracting symptoms (features)...")
tags_list = [[t.strip() for t in str(tags_str).split(',')] for tags_str in df['features']]

print("One-hot encoding dataset using MultiLabelBinarizer...")
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(tags_list)
all_symptoms = list(mlb.classes_)

print(f"Total unique symptoms found across all datasets: {len(all_symptoms)}")

y = df['label'].str.strip()

print("Encoding labels...")
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Splitting dataset into train and test sets...")
class_counts = y.value_counts()
valid_classes = class_counts[class_counts > 1].index

mask = y.isin(valid_classes)
X_filtered = X[mask]
y_filtered = y_encoded[mask]

if len(X_filtered) < 10:
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
else:
    X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered)

print("Training RandomForestClassifier...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Model accuracy on test set: {accuracy:.4f}")

print("Saving models...")
joblib.dump(model, 'classifier.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(all_symptoms, 'feature_list.pkl')
print("✅ All datasets trained and models saved successfully.")

In [ ]:
# Gunakan baris ini HANYA JIKA berjalan di Google Colab untuk mendownload hasilnya otomatis
try:
    from google.colab import files
    files.download('classifier.pkl')
    files.download('label_encoder.pkl')
    files.download('feature_list.pkl')
except ImportError:
    print("Script ini tidak berjalan di Google Colab. Silakan download file .pkl secara manual.")